In [27]:
import torch
import torch.nn as nn
from einops import rearrange

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.10.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


In [28]:

class CNN_Style_Encoder(nn.Module):
    """
    Scans handwriting with filters to extract stroke features
    Input: (B,1,64,256) grayscale image of a word
    Output: (B, 256) one feature vector per image
    """
    def __init__(self):
        super().__init__()

        #Learn the edges and structures of the handwriting
        # 64x256 -> 32x128
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.GroupNorm(8, 64),
            nn.SiLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.GroupNorm(8, 64),
            nn.SiLU(),
            nn.MaxPool2d(2),
        )

        #Learn letters shapes
        # 32x128 -> 16x64
        self.conv2 = nn.Sequential(
            nn.Conv2d(64,128, kernel_size=3, padding=1),
            nn.GroupNorm(8,128),
            nn.SiLU(),
            nn.Conv2d(128,128, kernel_size=3, padding=1),
            nn.GroupNorm(8,128),
            nn.SiLU(),
            nn.MaxPool2d(2),
        )

        #Learn writing stlyle and patterns
        #16x64 -> 4x16
        self.conv3 = nn.Sequential(
            nn.Conv2d(128,256, kernel_size=3, padding=1),
            nn.GroupNorm(8,256),
            nn.SiLU(),
            nn.Conv2d(256,256, kernel_size=3, padding=1),
            nn.GroupNorm(8,256),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.flatten = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*4*4, 256),
            nn.LayerNorm(256),
        )
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.flatten(x)
        return x

In [29]:
model = CNN_Style_Encoder()

x = torch.randn(4,1,64,256)  # Batch of 4 grayscale images
c1 = model.conv1(x)
c2 = model.conv2(c1)
c3 = model.conv3(c2)
out = model.flatten(c3)

print("Input  :", x.shape)    # (4, 1,   64, 256)
print("Conv1  :", c1.shape)   # (4, 64,  32, 128)
print("Conv2  :", c2.shape)   # (4, 128, 16,  64)
print("Conv3  :", c3.shape)   # (4, 256,  4,   4)
print("Output :", out.shape)  # (4, 256)

Input  : torch.Size([4, 1, 64, 256])
Conv1  : torch.Size([4, 64, 32, 128])
Conv2  : torch.Size([4, 128, 16, 64])
Conv3  : torch.Size([4, 256, 4, 4])
Output : torch.Size([4, 256])


In [ ]:

class Patch_Embedding(nn.Module):
    """
    Seperates images into patches and converts to it to an embedding vector
    Input : (B,1,64,256)
    Output : (B,256, 256)
    """
    def __init__(self,patch_size = 8, embed_dim = 256):
        super().__init__()
        self.patch_size = patch_size

        # Each patch is 8x8 pix x1 channel = 64 numbers
        # Linear projects 64 numbers -> 256 numbers (rich embedding)
        self.proj = nn.Linear(patch_size * patch_size * 1,embed_dim)
    
    def forward(self,x ):
        # cuts images into 8x8 patches
        B, C, H, W = x.shape
        p = self.patch_size
        # Reshape it into patch grid
        x = x.reshape(B, C, H//p, p, W//p, p)  # (B, C, num_patches_h, patch_size, num_patches_w, patch_size)
        # Reorder Dimensions to get patches as a sequence
        x = x.permute(0, 2,4, 3, 5)
        # flatten patch pixels
        x= x.reshape(B, (H//p)*(W//p), p*p*C) # (B, num_patches, patch_size* patch_size * C)
        return self.proj(x) # -> (B, num_patches, embed_dim)

embed = Patch_Embedding(patch_size= 8,embed_dim= 256)
x = torch.randn(4, 1, 64, 256)
out = embed(x)
print("Input:",x.shape)
print("Patches:",out.shape)

    


Input: torch.Size([4, 1, 64, 256])
Patches: torch.Size([4, 256, 256])


In [31]:
class VIT_Embeddings(nn.Module):
    """
    Adds CLS token and position info to patch embeddings
    Input: (B,256,256) patch embeddings
    Output: (B, 257, 256) patch embeddings + CLS token, with positions
    """
    def __init__(self, num_patches = 256, embed_dim = 256):
        super().__init__()
        # CLS token - a learnable summary vector
        # starts random and learns to represent the whole image
        self.cls_token = nn.Parameter(torch.randn(1,1,embed_dim)* 0.02)

        # Position embeddings - learnable vectors for each patch position
        # helps model understand the order of patches
        # 256 patches + 1 CLS token = 257 total tokens
        self.pos_embed = nn.Parameter(torch.randn(1,num_patches + 1, embed_dim)* 0.02)

    def forward(self,x):
        B = x.shape[0]

        # Expand CLS token for whole size
        cls = self.cls_token.expand(B, -1, -1)    # (1,1,256) → (B,1,256)

        # Prepare CLS to patch sequence
        x = torch.cat([cls, x], dim =1)           # (B,256,256) → (B,257,256)

        # Add position info to every token
        x = x + self.pos_embed                    # (B,257,256)
        return x
patch_embed = Patch_Embedding(patch_size= 8,embed_dim= 256)
vit_embed = VIT_Embeddings(num_patches= 256, embed_dim= 256)

x = torch.randn(4, 1, 64, 256)
patches = patch_embed(x)
out = vit_embed(patches)
print("Patches:",patches.shape) # (4, 256, 256)
print("VIT Embed:",out.shape)   # (4, 257, 256) 
print("CLS Token:", vit_embed.cls_token.shape) # (1, 1, 256)



Patches: torch.Size([4, 256, 256])
VIT Embed: torch.Size([4, 257, 256])
CLS Token: torch.Size([1, 1, 256])


In [32]:

class Transformer_Block(nn.Module):
    """
    One transformer block - patches are attended to each other and learn relationships
    Input: (B, 257, 256) patch embeddings + CLS token, with positions
    Output: (B, 257, 256) same shape but with learned relationships
    """
    def __init__(self, embed_dim = 256, num_heads = 8, dropout=0.1):
        super().__init__()
        # Normalization before attention
        self.norm1 = nn.LayerNorm(embed_dim)

        # Multi-head self attention - learns relationships between patches
        # num_heads = 8 we have 8 parallel attention layers to capture different relationships
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        # Normalization before MLP
        self.norm2 = nn.LayerNorm(embed_dim)

        # MLP - tokens processed independently to learn complex features
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),  # Expand to higher dimension
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim *4 ,embed_dim),
            nn.Dropout(dropout),
        )
    def forward(self,x):
        # Attention with residual connection
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out

        # MLP with residual connection
        x_norm = self.norm2(x)
        attn_out = self.mlp(x_norm)
        x = x + attn_out
        return x
    
# Test the transformer block
block = Transformer_Block(embed_dim =256, num_heads=8, dropout=0.1)
x = torch.randn(4, 257, 256)  # (B, tokens, embed_dim)
out = block(x)

print("Input:", x.shape)   # (4, 257, 256)
print("Output:", out.shape) # (4, 257, 256) 


Input: torch.Size([4, 257, 256])
Output: torch.Size([4, 257, 256])


In [33]:
class VIT_Style_Encoder(nn.Module):
    """
    Full Vision Transformer for handwriting style extraction.
    Understanding global letter flow and connections.
    Input: (B,1,64,256) - greyscale image
    Output: (B, 256) - style vector from CLS token
    """
    def __init__(self, patch_size=8, embed_dim=256, depth=6 ,num_heads=8, dropout =0.1):
        super().__init__()

        # cut images to patches and embed them
        self.patch_embed = Patch_Embedding(patch_size=patch_size, embed_dim=embed_dim)

        # add CLS token and position info
        self.vit_embed = VIT_Embeddings(num_patches= 256, embed_dim=embed_dim)

        # stack of transformer blocks to learn relationships between patches
        self.blocks = nn.Sequential(*[Transformer_Block(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout)
                                      for _ in range(depth)
        ])
        
        # Normalize final output
        self.norm = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        # patches and embed
        x = self.patch_embed(x)  # (B, 256, 256)
        
        # Add CLS token and position 
        x = self.vit_embed(x)  # (B, 257, 256)

        # Run through 6 transformer blocks
        x = self.blocks(x)  # (B, 257, 256)

        # Normalize 
        x = self.norm(x)  # (B, 257, 256)

        # Return CLS token as style vector
        return x[:,0,:]  # (B, 256)
# TEST IT ;)
vit = VIT_Style_Encoder(patch_size=8, embed_dim=256, depth=6, num_heads=8)
x = torch.randn(4, 1, 64, 256)
out = vit(x)

print("Input:", x.shape)  # (4, 1, 64, 256)
print("Output:", out.shape) # (4, 256) - CLS token style vector 
print(f"Params: {sum(p.numel() for p in vit.parameters()):,}")



Input: torch.Size([4, 1, 64, 256])
Output: torch.Size([4, 256])
Params: 4,821,760


In [36]:
class Style_Encoder(nn.Module):
    """
    Full style extractor combines CNN local features and VIT global features
    Input: (B, 1, 64, 256) - greyscale handwriting word
    Output: (B, 256) - final style vector 
    """ 
    def __init__(self):
        super().__init__()

        # CNN extracts local stroke features
        self.cnn = CNN_Style_Encoder()

        # ViT captures global style and letter flow
        self.vit = VIT_Style_Encoder()

        # Fusion layer to combine CNN and ViT features
        self.fusion = nn.Sequential(
            nn.Linear(512, 512),  # CNN(256) + ViT(256) -> 512
            nn.LayerNorm(512),
            nn.SiLU(),
        )
    def forward(self, x):

        cnn_features = self.cnn(x)
        vit_features = self.vit(x)

        combined = torch.cat([cnn_features, vit_features], dim = 1) # (B, 512)
        style =  self.fusion(combined)

        return style
# Test it ;]
encoder = Style_Encoder()
x = torch.randn(4, 1, 64, 256)
out = encoder(x)

print("Input :", x.shape)    # (4, 1, 64, 256)
print("Output:", out.shape)  # (4, 512)
print(f"Total Params: {sum(p.numel() for p in encoder.parameters()):,}")


Input : torch.Size([4, 1, 64, 256])
Output: torch.Size([4, 512])
Total Params: 7,280,832
